# 10 — Portfolio Construction

**AFML Chapter 16** — López de Prado (2018)

The capstone notebook.  We combine everything built in this series:
- Triple-barrier labels + meta-labeling → position signals
- Feature importance → feature selection
- Purged CV → unbiased model evaluation
- **HRP** → portfolio allocation across multiple crypto assets
- Deflated Sharpe → overfitting check

Hierarchical Risk Parity (HRP) allocates risk through a dendrogram,
avoiding the instability of mean-variance optimisation.

---

In [ ]:
from _setup import *
from afml.portfolio import hrp_weights, inverse_variance_weights, equal_weights, correlation_distance
from afml.backtest_stats import sharpe_ratio, return_stats, deflated_sharpe_ratio, expected_max_sharpe
from afml.labeling import daily_volatility

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

## 1. Build multi-asset return matrix

In [ ]:
panel = load_daily_bars()

# Select top assets by data availability
sym_counts = panel.groupby("symbol")["ts"].count().sort_values(ascending=False)
top_symbols = sym_counts.head(15).index.tolist()

# Build return matrix
closes = {}
for sym in top_symbols:
    asset = panel[panel["symbol"] == sym].sort_values("ts").drop_duplicates("ts", keep="last")
    closes[sym] = asset.set_index("ts")["close"]

price_df = pd.DataFrame(closes)
# Use common date range (at least 500 days of data)
min_obs = 500
valid_cols = price_df.columns[price_df.count() >= min_obs]
price_df = price_df[valid_cols].dropna()

returns = np.log(price_df / price_df.shift(1)).dropna()

print(f"Universe: {len(returns.columns)} assets")
print(f"Period: {returns.index[0].date()} to {returns.index[-1].date()} ({len(returns)} days)")
print(f"Assets: {list(returns.columns)}")

## 2. Correlation structure and dendrogram

In [ ]:
corr = returns.corr()
dist = correlation_distance(corr)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Correlation heatmap
import seaborn as sns
ax = axes[0]
sns.heatmap(corr, ax=ax, cmap="RdBu_r", center=0, vmin=-0.3, vmax=1,
            square=True, linewidths=0.5, fmt=".2f",
            xticklabels=True, yticklabels=True)
ax.set_title("Return correlation matrix", fontweight="bold")

# Dendrogram
ax = axes[1]
dist_condensed = squareform(dist.values, checks=False)
link = linkage(dist_condensed, method="single")
dendrogram(link, labels=list(returns.columns), ax=ax, leaf_rotation=45)
ax.set_title("Hierarchical clustering (single linkage)", fontweight="bold")
ax.set_ylabel("Correlation distance")

fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "10_correlation_dendrogram.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. HRP vs alternatives

In [ ]:
w_hrp = hrp_weights(returns, method="single")
w_ivp = inverse_variance_weights(returns)
w_eq = equal_weights(returns)

weights_df = pd.DataFrame({"HRP": w_hrp, "Inv-Var": w_ivp, "Equal": w_eq})
display(weights_df.style.format("{:.1%}").bar(color=NAVY, subset=["HRP"]))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(weights_df))
width = 0.25

ax.bar(x - width, weights_df["HRP"], width, color=NAVY, alpha=0.7, label="HRP")
ax.bar(x, weights_df["Inv-Var"], width, color=TEAL, alpha=0.7, label="Inverse Variance")
ax.bar(x + width, weights_df["Equal"], width, color=GOLD, alpha=0.7, label="Equal Weight")

ax.set_xticks(x)
ax.set_xticklabels(weights_df.index, rotation=45, ha="right")
ax.set_ylabel("Weight")
ax.set_title("Portfolio weights: HRP vs Inverse Variance vs Equal", fontweight="bold")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "10_weights.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Backtest: walk-forward portfolio performance

In [ ]:
# Walk-forward: rebalance quarterly using trailing 1-year window
rebal_freq = 63  # ~quarterly
lookback = 252

portfolio_returns = {"HRP": [], "Inv-Var": [], "Equal": []}
dates_out = []

for t in range(lookback, len(returns) - 1, rebal_freq):
    train = returns.iloc[t - lookback : t]
    end = min(t + rebal_freq, len(returns))
    test = returns.iloc[t : end]

    w_h = hrp_weights(train, method="single")
    w_i = inverse_variance_weights(train)
    w_e = equal_weights(train)

    for method, w in [("HRP", w_h), ("Inv-Var", w_i), ("Equal", w_e)]:
        port_ret = (test * w).sum(axis=1)
        portfolio_returns[method].extend(port_ret.values.tolist())

    dates_out.extend(test.index.tolist())

# Build cumulative returns
perf = pd.DataFrame(portfolio_returns, index=dates_out[:len(portfolio_returns["HRP"])])
cum = (1 + perf).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={"height_ratios": [2, 1]})

ax = axes[0]
for col, color in zip(["HRP", "Inv-Var", "Equal"], [NAVY, TEAL, GOLD]):
    ax.plot(cum.index, cum[col], color=color, lw=1.2, label=col)
ax.set_ylabel("Cumulative return")
ax.set_title("Walk-forward portfolio backtest (quarterly rebalance)", fontweight="bold")
ax.legend(fontsize=10)

# Rolling Sharpe
ax = axes[1]
roll_window = 126
for col, color in zip(["HRP", "Inv-Var", "Equal"], [NAVY, TEAL, GOLD]):
    roll_sr = perf[col].rolling(roll_window).apply(
        lambda x: x.mean() / x.std() * np.sqrt(252) if x.std() > 0 else 0
    )
    ax.plot(roll_sr.index, roll_sr, color=color, lw=0.8, label=col)

ax.axhline(0, color=GRAY, ls="--", lw=0.5)
ax.set_ylabel("Rolling Sharpe (126d)")
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "10_backtest.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Performance summary + Deflated Sharpe check

In [ ]:
summary = []
for method in ["HRP", "Inv-Var", "Equal"]:
    rets_m = perf[method]
    stats = return_stats(rets_m)
    ann_ret = rets_m.mean() * 252
    ann_vol = rets_m.std() * np.sqrt(252)
    max_dd = (cum[method] / cum[method].cummax() - 1).min()

    # DSR with 3 trials (we tested 3 methods)
    benchmark = expected_max_sharpe(3)
    dsr = deflated_sharpe_ratio(
        observed_sr=stats["sharpe"],
        sr_benchmark=benchmark,
        n_obs=stats["n_obs"],
        skewness=stats["skewness"],
        excess_kurtosis=stats["excess_kurtosis"],
    )

    summary.append({
        "Method": method,
        "Ann. Return": ann_ret,
        "Ann. Vol": ann_vol,
        "Sharpe": stats["sharpe"],
        "Max DD": max_dd,
        "Skewness": stats["skewness"],
        "Kurtosis": stats["excess_kurtosis"],
        "DSR (3 trials)": dsr,
    })

df_summary = pd.DataFrame(summary).set_index("Method")
display(df_summary.style.format({
    "Ann. Return": "{:.1%}",
    "Ann. Vol": "{:.1%}",
    "Sharpe": "{:.2f}",
    "Max DD": "{:.1%}",
    "Skewness": "{:.2f}",
    "Kurtosis": "{:.1f}",
    "DSR (3 trials)": "{:.1%}",
}))

## 6. Summary & Series Conclusion

### HRP key findings:
- HRP produces more diversified weights than equal/inverse-variance
- Allocates less to correlated assets (clusters them together first)
- Doesn't require inverting the covariance matrix → more stable

### Complete AFML implementation:

| # | Notebook | AFML Chapter | Module |
|---|----------|-------------|--------|
| 01 | Data Structures | Ch 2 | `afml.bars` |
| 02 | Labeling | Ch 3 | `afml.labeling` |
| 03 | Fractional Differentiation | Ch 5 | `afml.fracdiff` |
| 04 | Cross-Validation | Ch 7, 12 | `afml.cross_validation` |
| 05 | Feature Importance | Ch 8 | `afml.feature_importance` |
| 06 | Bet Sizing | Ch 10 | `afml.bet_sizing` |
| 07 | Structural Breaks & Entropy | Ch 14–15 | `afml.features` |
| 08 | Microstructure | Ch 18–19 | `afml.microstructure` |
| 09 | Backtesting Dangers | Ch 11 | `afml.backtest_stats` |
| 10 | Portfolio Construction | Ch 16 | `afml.portfolio` |

All implementations use the full crypto universe and are integrated
with the existing backtesting framework.